# Latent utility Stage 0 — Hiera-B+ sanity

Non-claim-bearing preflight for preregistration v2. It verifies the exact B+ model, frozen protocol, Kaggle GPU, W&B Forge connection, DAVIS resolution, no-op intervention, cached-state restoration, and attention reconstruction before any test-set evidence is counted.

In [ ]:
import importlib.util, json, os, shutil, subprocess, sys, time
from pathlib import Path
from kaggle_secrets import UserSecretsClient

WORK = Path('/kaggle/working')
REPO = WORK / 'AdeSEG'
OUT = WORK / 'latent_utility_stage0'
OUT.mkdir(parents=True, exist_ok=True)
started = time.time()

def mark(stage, state='running', **details):
    payload = {'stage': stage, 'state': state, 'elapsed_seconds': time.time() - started, **details}
    temporary = OUT / 'status.json.tmp'
    temporary.write_text(json.dumps(payload, indent=2) + '\n')
    os.replace(temporary, OUT / 'status.json')
    with (OUT / 'events.jsonl').open('a') as stream:
        stream.write(json.dumps(payload, sort_keys=True) + '\n')
    print('[status]', json.dumps(payload, sort_keys=True), flush=True)

def run(command, cwd=None, timeout=900):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    try:
        subprocess.run(command, cwd=cwd, env=os.environ.copy(), check=True, timeout=timeout)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        mark('command_failed', state='failed', command=' '.join(command), error=repr(exc))
        raise

mark('bootstrap')
wandb_api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
assert wandb_api_key, 'Attach the Kaggle secret WANDB_API_KEY to this notebook before running'
os.environ.update({'WANDB_API_KEY': wandb_api_key, 'WANDB_BASE_URL': 'https://forge.coreweave.com/api/wandb', 'WANDB_PROJECT': 'ade-seg', 'WANDB_ENTITY': 'adenoid-hypertrophy', 'PYTHONUNBUFFERED': '1', 'GIT_TERMINAL_PROMPT': '0'})
mark('secret_loaded')

In [ ]:
mark('repository_setup')
if REPO.exists():
    shutil.rmtree(REPO)
run(['git', 'clone', '--depth', '1', '--branch', 'experiment/utility-aware-memory-2026-10-16', 'https://github.com/aCoderChild/AdeSEG.git', REPO], timeout=300)
missing = [package for module, package in {'hydra': 'hydra-core>=1.3.2', 'iopath': 'iopath>=0.1.10', 'wandb': 'wandb'}.items() if importlib.util.find_spec(module) is None]
if missing:
    run([sys.executable, '-m', 'pip', 'install', *missing], timeout=300)
if importlib.util.find_spec('tensordict') is None:
    run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'tensordict>=0.7,<0.8'], timeout=300)
import torch
assert torch.cuda.is_available(), 'Stage 0 requires a Kaggle GPU; CPU fallback is intentionally disabled'
run([sys.executable, REPO/'scripts/verify_wandb_connection.py', '--name', 'latent-utility-stage0-connectivity'], cwd=REPO, timeout=120)
mark('repository_ready', commit=subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip(), torch=torch.__version__, accelerator=torch.cuda.get_device_name(0))

In [ ]:
mark('data_and_checkpoint')
PROTOCOL = REPO / 'protocols/latent_utility_v2.json'
DATA = WORK / 'experiment_data'
DATA.mkdir(exist_ok=True)
DAVIS_MANIFEST = DATA / 'davis.jsonl'
run([sys.executable, REPO/'scripts/build_davis_manifest.py', '--data-root', '/kaggle/input', '--protocol', PROTOCOL, '--output', DAVIS_MANIFEST], cwd=REPO, timeout=120)
split_file = json.loads(PROTOCOL.read_text())['datasets']['davis_train']['sequence_partition_file']
SMOKE_VIDEO = json.loads((PROTOCOL.parent/split_file).read_text())['davis_2017']['validation_sequences'][0]
CKPT = DATA / 'sam2.1_hiera_base_plus.pt'
run(['wget', '--quiet', '--show-progress', '--timeout=30', '--tries=3', 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt', '-O', CKPT], timeout=600)
assert CKPT.stat().st_size > 300_000_000, CKPT.stat().st_size
mark('inputs_ready', smoke_video=SMOKE_VIDEO, checkpoint_bytes=CKPT.stat().st_size)

In [ ]:
mark('bplus_sanity')
SMOKE = OUT / 'davis_val_one_frame'
run([sys.executable, REPO/'scripts/generate_utility_labels.py', '--sam2-cfg', 'configs/sam2.1_hiera_b+1024.yaml', '--sam2-checkpoint', CKPT, '--manifest', DAVIS_MANIFEST, '--split', 'val', '--video-ids', SMOKE_VIDEO, '--label-ids', '1', '--protocol', PROTOCOL, '--output-dir', SMOKE, '--device', 'cuda', '--max-analyzed-frames', '1', '--frame-selection', 'every_n', '--every-n', '20', '--wandb-base-url', os.environ['WANDB_BASE_URL'], '--wandb-name', 'latent-utility-stage0-bplus'], cwd=REPO, timeout=10800)
summary = json.loads((SMOKE/'summary.json').read_text())
status = json.loads((SMOKE/'status.json').read_text())
assert summary['analyzed_frames'] == 1, summary
assert status['state'] == 'complete', status
assert summary['sanity_no_op_max_abs'] == 0.0, summary
assert summary['sanity_cached_state_restore_max_abs'] == 0.0, summary
assert summary['sanity_attention_reconstruction_max_abs'] is not None, summary
assert summary['sanity_attention_reconstruction_max_abs'] < 1e-5, summary
mark('complete', state='complete', analyzed_frames=1, no_op_max_abs=summary['sanity_no_op_max_abs'], cached_restore_max_abs=summary['sanity_cached_state_restore_max_abs'], attention_reconstruction_max_abs=summary['sanity_attention_reconstruction_max_abs'])
print('STAGE 0 COMPLETE:', json.dumps(summary, indent=2), flush=True)